In [1]:
import os

import pandas as pd
import geopandas as gpd
from tqdm import tqdm

In [2]:
# CELL 2 — Constants: file paths, ward name mapping, and characteristic IDs

RAW_DIR = "../../data/census/wards/98-401-X2021010_eng_CSV"
GEO_INDEX_FILE = f"{RAW_DIR}/98-401-X2021010_Geo_starting_row.CSV"
MAIN_FILE = f"{RAW_DIR}/98-401-X2021010_English_CSV_data.csv"

OUTPUT_DIR = "../../data/census/wards/toronto"
FINAL_OUTPUT = "../../data/census/wards/toronto-wards.csv"

# This census product (98-401-X2021010) is published at the FEDERAL ELECTORAL
# DISTRICT level, not at the municipal ward level directly. However, Toronto's
# 25 municipal wards were redrawn in 2018 to match the 25 federal electoral
# district boundaries within the city exactly, so each riding below is a 1:1
# stand-in for the correspondingly named ward.
#
# Maps the GEO_NAME spelling used in the raw census CSV (federal riding,
# double-dash separated) to the ward_name spelling used elsewhere in this repo
# (see data/geo/city-wards.geojson and data/activity/ward_to_venue_summary.csv),
# which is the join key we want downstream.
TORONTO_WARDS = {
    "Beaches--East York": "Beaches-East York",
    "Davenport": "Davenport",
    "Don Valley East": "Don Valley East",
    "Don Valley North": "Don Valley North",
    "Don Valley West": "Don Valley West",
    "Eglinton--Lawrence": "Eglinton-Lawrence",
    "Etobicoke Centre": "Etobicoke Centre",
    "Etobicoke--Lakeshore": "Etobicoke-Lakeshore",
    "Etobicoke North": "Etobicoke North",
    "Humber River--Black Creek": "Humber River-Black Creek",
    "Parkdale--High Park": "Parkdale-High Park",
    "Scarborough--Agincourt": "Scarborough-Agincourt",
    "Scarborough Centre": "Scarborough Centre",
    "Scarborough--Guildwood": "Scarborough-Guildwood",
    "Scarborough North": "Scarborough North",
    "Scarborough--Rouge Park": "Scarborough-Rouge Park",
    "Scarborough Southwest": "Scarborough Southwest",
    "Spadina--Fort York": "Spadina-Fort York",
    "Toronto Centre": "Toronto Centre",
    "Toronto--Danforth": "Toronto-Danforth",
    "Toronto--St. Paul's": "Toronto-St. Paul's",
    "University--Rosedale": "University-Rosedale",
    "Willowdale": "Willowdale",
    "York Centre": "York Centre",
    "York South--Weston": "York South-Weston",
}

# ward_name -> two-digit ward_code, from data/geo/city-wards.geojson
WARD_CODES = {
    "Etobicoke North": "01",
    "Etobicoke Centre": "02",
    "Etobicoke-Lakeshore": "03",
    "Parkdale-High Park": "04",
    "York South-Weston": "05",
    "York Centre": "06",
    "Humber River-Black Creek": "07",
    "Eglinton-Lawrence": "08",
    "Davenport": "09",
    "Spadina-Fort York": "10",
    "University-Rosedale": "11",
    "Toronto-St. Paul's": "12",
    "Toronto Centre": "13",
    "Toronto-Danforth": "14",
    "Don Valley West": "15",
    "Don Valley East": "16",
    "Don Valley North": "17",
    "Willowdale": "18",
    "Beaches-East York": "19",
    "Scarborough Southwest": "20",
    "Scarborough Centre": "21",
    "Scarborough-Agincourt": "22",
    "Scarborough North": "23",
    "Scarborough-Guildwood": "24",
    "Scarborough-Rouge Park": "25",
}

# Mapping from CHARACTERISTIC_ID to semantic short codes, kept identical to
# analysis/census/extract_census_ada.ipynb so ward-level and ADA-level
# extracts line up on the same variables for later comparison/aggregation.
CHARACTERISTIC_CODES = {
    # Demographics - Population
    1: 'pop_2021',  # Population, 2021
    2: 'pop_2016',  # Population, 2016
    3: 'pop_pct_change',  # Population % change, 2016 to 2021
    6: 'pop_density',  # Population density per square kilometre

    # Age structure
    34: 'age_total_dist',  # Total - Distribution (%) of the population by broad age groups
    35: 'age_0_14',  # 0 to 14 years
    36: 'age_15_64',  # 15 to 64 years
    37: 'age_65_over',  # 65 years and over
    38: 'age_85_over',  # 85 years and over
    39: 'age_mean',  # Average age of the population
    40: 'age_median',  # Median age of the population

    # Household composition
    50: 'pvt_house_total',  # Total - Private households by household size
    51: 'pvt_house_1',  # 1 person
    52: 'pvt_house_2',  # 2 persons
    53: 'pvt_house_3',  # 3 persons
    54: 'pvt_house_4',  # 4 persons
    55: 'pvt_house_5plus',  # 5 or more persons
    56: 'pvt_house_persons',  # Number of persons in private households
    57: 'pvt_house_avg_size',  # Average household size

    # Income - Core measures
    111: 'income_total',  # Total - Income statistics (population aged 15+)
    112: 'income_total_recip',  # Number of total income recipients
    113: 'income_total_median',  # Median total income among recipients ($)
    114: 'income_aftertax_recip',  # Number of after-tax income recipients
    115: 'income_aftertax_median',  # Median after-tax income among recipients ($)
    116: 'income_market_recip',  # Number of market income recipients
    117: 'income_market_median',  # Median market income among recipients ($)
    118: 'income_emp_recip',  # Number of employment income recipients
    119: 'income_emp_median',  # Median employment income among recipients ($)
    120: 'income_govtrans_recip',  # Number of government transfers recipients
    121: 'income_govtrans_median',  # Median government transfers among recipients ($)
    122: 'income_ei_recip',  # Number of employment insurance benefits recipients
    123: 'income_ei_median',  # Median employment insurance benefits among recipients ($)
    124: 'income_covid_recip',  # Number of COVID-19 emergency and recovery benefits recipients
    125: 'income_covid_median',  # Median COVID-19 emergency and recovery benefits ($)

    # Income - Poverty and distribution
    345: 'lim_at_prev',  # Prevalence of low income based on LIM-AT (%)
    346: 'lim_at_0_17',  # LIM-AT prevalence, age 0 to 17 (%)
    347: 'lim_at_0_5',  # LIM-AT prevalence, age 0 to 5 (%)
    348: 'lim_at_18_64',  # LIM-AT prevalence, age 18 to 64 (%)
    349: 'lim_at_65_over',  # LIM-AT prevalence, age 65 years and over (%)
    365: 'income_decile_total',  # Total - Adjusted after-tax economic family income decile
    366: 'income_decile_bottom_half',  # In bottom half of the distribution
    367: 'income_decile_1',  # In bottom decile
    368: 'income_decile_2',  # In second decile
    369: 'income_decile_3',  # In third decile
    370: 'income_decile_4',  # In fourth decile
    371: 'income_decile_5',  # In fifth decile
    372: 'income_decile_top_half',  # In top half of the distribution
    373: 'income_decile_6',  # In sixth decile
    374: 'income_decile_7',  # In seventh decile
    375: 'income_decile_8',  # In eighth decile
    376: 'income_decile_9',  # In ninth decile
    377: 'income_decile_10',  # In top decile
    378: 'gini_measures',  # Total - Inequality measures (population count)
    379: 'gini_total_income',  # Gini index on adjusted household total income
    380: 'gini_market_income',  # Gini index on adjusted household market income
    381: 'gini_aftertax_income',  # Gini index on adjusted household after-tax income
    382: 'gini_p90_p10',  # P90/P10 ratio on adjusted household after-tax income

    # Housing - Tenure and affordability
    1414: 'housing_tenure_total',  # Total - Private households by tenure
    1415: 'housing_tenure_owner',  # Owner
    1416: 'housing_tenure_renter',  # Renter
    1417: 'housing_tenure_provided',  # Dwelling provided by local government, First Nation or Indian band
    1418: 'housing_condo_total',  # Total - Occupied private dwellings by condominium status
    1465: 'housing_shelter_total',  # Total - Owner and tenant households by shelter-cost-to-income ratio
    1466: 'housing_shelter_under30',  # Spending less than 30% of income on shelter costs
    1467: 'housing_shelter_30plus',  # Spending 30% or more of income on shelter costs
    1468: 'housing_shelter_30_100',  # Spending 30% to less than 100% of income on shelter costs

    # Housing - Core need
    1479: 'housing_core_need_total',  # Total - Owner and tenant households (core need denominator)
    1480: 'housing_core_need_yes',  # In core housing need
    1481: 'housing_core_need_no',  # Not in core housing need
    1482: 'housing_owner_total',  # Total - Owner households in non-farm, non-reserve private dwellings
    1483: 'housing_owner_mortgage_pct',  # % of owner households with a mortgage
    1484: 'housing_owner_shelter_30plus_pct',  # % of owner households spending 30%+ on shelter costs
    1485: 'housing_owner_core_need_pct',  # % of owners in core housing need

    # Citizenship
    1522: 'citizen_total',  # Total - Citizenship for the population in private households
    1523: 'citizen_canadian',  # Canadian citizens
    1524: 'citizen_canadian_under18',  # Canadian citizens aged under 18
    1525: 'citizen_canadian_18over',  # Canadian citizens aged 18 and over
    1526: 'citizen_not_canadian',  # Not Canadian citizens

    # Visible minorities
    1683: 'visible_minority_total',  # Total - Visible minority for the population in private households
    1684: 'visible_minority_yes',  # Total visible minority population
    1685: 'visible_minority_south_asian',  # South Asian
    1686: 'visible_minority_chinese',  # Chinese
    1687: 'visible_minority_black',  # Black
    1688: 'visible_minority_filipino',  # Filipino
    1689: 'visible_minority_arab',  # Arab
    1690: 'visible_minority_latin_american',  # Latin American
    1691: 'visible_minority_southeast_asian',  # Southeast Asian
    1692: 'visible_minority_west_asian',  # West Asian
    1693: 'visible_minority_korean',  # Korean
    1694: 'visible_minority_japanese',  # Japanese
    1695: 'visible_minority_nie',  # Visible minority, n.i.e.
    1696: 'visible_minority_multiple',  # Multiple visible minorities
    1697: 'visible_minority_no',  # Not a visible minority

    # Education - all ages 15+
    1998: 'education_total',  # Total - Highest certificate, diploma or degree (age 15+)
    1999: 'education_none',  # No certificate, diploma or degree
    2000: 'education_secondary',  # High (secondary) school diploma or equivalency certificate
    2001: 'education_postsec',  # Postsecondary certificate, diploma or degree
    2002: 'education_postsec_below_bachelor',  # Postsecondary certificate or diploma below bachelor level
    2003: 'education_apprentice_trades',  # Apprenticeship or trades certificate or diploma
    2004: 'education_apprentice_non',  # Non-apprenticeship trades certificate or diploma
    2005: 'education_apprentice_cert',  # Apprenticeship certificate
    2006: 'education_college',  # College, CEGEP or other non-university certificate or diploma
    2007: 'education_university_below_bachelor',  # University certificate or diploma below bachelor level
    2008: 'education_bachelor_higher',  # Bachelor's degree or higher
    2009: 'education_bachelor',  # Bachelor's degree
    2010: 'education_university_above_bachelor',  # University certificate or diploma above bachelor level
    2011: 'education_medical_dental_vet',  # Degree in medicine, dentistry, veterinary medicine or optometry
    2012: 'education_masters',  # Master's degree
    2013: 'education_doctorate',  # Earned doctorate degree

    # Education - ages 25 to 64
    2014: 'education_25_64_total',  # Total - Highest certificate, diploma or degree (age 25-64)
    2015: 'education_25_64_none',  # No certificate, diploma or degree (age 25-64)
    2016: 'education_25_64_secondary',  # High school diploma or equivalency (age 25-64)
    2017: 'education_25_64_postsec',  # Postsecondary certificate, diploma or degree (age 25-64)
    2018: 'education_25_64_postsec_below_bachelor',  # Postsecondary below bachelor level (age 25-64)
    2019: 'education_25_64_apprentice_trades',  # Apprenticeship or trades certificate (age 25-64)
    2020: 'education_25_64_apprentice_non',  # Non-apprenticeship trades certificate (age 25-64)
    2021: 'education_25_64_apprentice_cert',  # Apprenticeship certificate (age 25-64)
    2022: 'education_25_64_college',  # College, CEGEP or non-university certificate (age 25-64)
    2023: 'education_25_64_university_below_bachelor',  # University certificate below bachelor (age 25-64)
    2024: 'education_25_64_bachelor_higher',  # Bachelor's degree or higher (age 25-64)
    2025: 'education_25_64_bachelor',  # Bachelor's degree (age 25-64)
    2026: 'education_25_64_university_above_bachelor',  # University certificate above bachelor (age 25-64)
    2027: 'education_25_64_medical_dental_vet',  # Degree in medicine, dentistry or veterinary (age 25-64)
    2028: 'education_25_64_masters',  # Master's degree (age 25-64)
    2029: 'education_25_64_doctorate',  # Earned doctorate degree (age 25-64)

    # Mobility - Mobility status 5 years ago
    1983: 'mobility_total',  # Total - Mobility status 5 years ago
    1984: 'mobility_non_movers',  # Non-movers
    1985: 'mobility_movers',  # Movers
    1986: 'mobility_non_migrants',  # Non-migrants
    1987: 'mobility_migrants',  # Migrants
    1988: 'mobility_internal_migrants',  # Internal migrants
    1989: 'mobility_intraprovincial_migrants',  # Intraprovincial migrants
    1990: 'mobility_interprovincial_migrants',  # Interprovincial migrants
    1991: 'mobility_external_migrants',  # External migrants

    # Labour - Occupation
    2246: 'labour_occupation_total',  # Total - Labour force by occupation
    2247: 'labour_occupation_na',  # Occupation - not applicable
    2248: 'labour_occupation_all',  # All occupations
    2249: 'labour_occupation_0_management',  # 0 Legislative and senior management
    2250: 'labour_occupation_1_business_finance',  # 1 Business, finance and administration
    2251: 'labour_occupation_2_science',  # 2 Natural and applied sciences and related
    2252: 'labour_occupation_3_health',  # 3 Health occupations
    2253: 'labour_occupation_4_education_law_social',  # 4 Education, law and social, community services
    2254: 'labour_occupation_5_arts_culture',  # 5 Arts, culture, recreation and sport
    2255: 'labour_occupation_6_sales_service',  # 6 Sales and service occupations
    2256: 'labour_occupation_7_trades_transport',  # 7 Trades, transport and equipment operators
    2257: 'labour_occupation_8_natural_resources',  # 8 Natural resources, agriculture and related
    2258: 'labour_occupation_9_manufacturing',  # 9 Occupations in manufacturing and utilities

    # Labour - Industry
    2259: 'labour_industry_total',  # Total - Labour force by industry sector
    2260: 'labour_industry_na',  # Industry - not applicable
    2261: 'labour_industry_all',  # All industries
    2262: 'labour_industry_11_agriculture',  # 11 Agriculture, forestry, fishing and hunting
    2263: 'labour_industry_21_mining',  # 21 Mining, quarrying, and oil and gas extraction
    2264: 'labour_industry_22_utilities',  # 22 Utilities
    2265: 'labour_industry_23_construction',  # 23 Construction
    2266: 'labour_industry_31_33_manufacturing',  # 31-33 Manufacturing
    2267: 'labour_industry_41_wholesale',  # 41 Wholesale trade
    2268: 'labour_industry_44_45_retail',  # 44-45 Retail trade
    2269: 'labour_industry_48_49_transportation',  # 48-49 Transportation and warehousing
    2270: 'labour_industry_51_information',  # 51 Information and cultural industries
    2271: 'labour_industry_52_finance',  # 52 Finance and insurance
    2272: 'labour_industry_53_real_estate',  # 53 Real estate and rental and leasing
    2273: 'labour_industry_54_professional',  # 54 Professional, scientific and technical services
    2274: 'labour_industry_55_management',  # 55 Management of companies and enterprises
    2275: 'labour_industry_56_administrative',  # 56 Administrative and support, waste management
    2276: 'labour_industry_61_education',  # 61 Educational services
    2277: 'labour_industry_62_healthcare',  # 62 Health care and social assistance
    2278: 'labour_industry_71_arts',  # 71 Arts, entertainment and recreation
    2279: 'labour_industry_72_accommodation',  # 72 Accommodation and food services
    2280: 'labour_industry_81_other_services',  # 81 Other services (except public administration)
    2281: 'labour_industry_91_public_admin',  # 91 Public administration

    # Transit - Main mode of commuting
    2603: 'commute_mode_total',  # Total - Main mode of commuting (employed labour force, usual workplace or no fixed address)
    2604: 'commute_mode_car_truck_van',  # Car, truck or van
    2605: 'commute_mode_car_driver',  # Car, truck or van - as a driver
    2606: 'commute_mode_car_passenger',  # Car, truck or van - as a passenger
    2607: 'commute_mode_transit',  # Public transit
    2608: 'commute_mode_walked',  # Walked
    2609: 'commute_mode_bicycle',  # Bicycle
    2610: 'commute_mode_other',  # Other method

    # Transit - Commuting duration
    2611: 'commute_duration_total',  # Total - Commuting duration (employed labour force, usual workplace or no fixed address)
    2612: 'commute_duration_under15',  # Less than 15 minutes
    2613: 'commute_duration_15_29',  # 15 to 29 minutes
    2614: 'commute_duration_30_44',  # 30 to 44 minutes
    2615: 'commute_duration_45_59',  # 45 to 59 minutes
    2616: 'commute_duration_60plus',  # 60 minutes and over
}

# CHARACTERISTIC_ID ranges (inclusive) by category — identical set to the ADA extract
ID_RANGES = [
    (1, 2), (3, 3), (6, 6),
    (34, 37), (38, 40),
    (50, 55), (56, 57),
    (111, 119), (120, 121), (122, 125),
    (345, 349), (365, 382),
    (1414, 1418), (1465, 1468),
    (1479, 1485),
    (1998, 2029),
    (1522, 1526),
    (1983, 1991),
    (1683, 1697),
    (2246, 2258),
    (2259, 2281),
    (2603, 2610), (2611, 2616),
]

# ---- NOC/NAICS labour-force bonus data (see section below) ----
# Tract-level counts + geometries, prepared upstream by analysis/census/interpolate_noc_naics.ipynb
# and its predecessor extract; census_tract is the join key, geometry lets us do the
# area interpolation ourselves rather than reusing that notebook's ADA-level output.
NOC_NAICS_TRACT_GEOJSON = "../../data/census/ada-wide/toronto-tract-noc-naics.geojson"
WARDS_GEOJSON = "../../data/geo/city-wards.geojson"
WARD_NAME_COL = "ward_name"  # column name in city-wards.geojson
NOC_NAICS_OUTPUT = "../../data/census/wards/toronto-wards-noc-naics.csv"

# The tract file carries both counts (extensive -- scale with population) and
# percents (intensive -- derived, don't interpolate directly). We interpolate the
# counts and recompute the percents at the ward level so the result is population-
# weighted rather than area-weighted (see markdown below for why that distinction matters).
NOC_NAICS_COUNT_FIELDS = [
    "pop_2021_count",
    "labour_creatives_count",
    "labour_cultural_workers_count",
    "labour_cultural_industries_count",
    "labour_independent_artists_count",
    "labour_arts_major_count",
]

# (numerator_count_field -> output_pct_field), all computed relative to pop_2021_count
NOC_NAICS_PCT_FIELDS = {
    "labour_creatives_count": "labour_creatives_pct",
    "labour_cultural_workers_count": "labour_cultural_workers_pct",
    "labour_cultural_industries_count": "labour_cultural_industries_pct",
    "labour_independent_artists_count": "labour_independent_artists_pct",
    "labour_arts_major_count": "labour_arts_major_pct",
}

## Verify the 25 Toronto wards against the geography index

Unlike the ADA extract, `GEO_NAME` in this product is a federal riding name
rather than a numeric geo code, and Toronto's ridings are scattered
alphabetically through the Ontario block rather than sitting in one
contiguous run — so we can't just match on a prefix. `98-401-X2021010_Geo_starting_row.CSV`
still tells us where each riding's rows start; here we look each Toronto
riding up in that index purely as a sanity check that our name list is
correct and to see the row ranges before we do the real (name-based) filter
below.

In [3]:
# CELL 3 — Load the Geo Starting Row index and locate the 25 Toronto wards

geo_index = pd.read_csv(GEO_INDEX_FILE, dtype=str, encoding="latin")
geo_index["Line Number"] = geo_index["Line Number"].astype(int)
geo_index = geo_index.sort_values("Line Number").reset_index(drop=True)

# End row of each geography's block = start row of the next geography (EOF for the last one)
geo_index["End Line"] = geo_index["Line Number"].shift(-1)

toronto_geo = geo_index[geo_index["Geo Name"].isin(TORONTO_WARDS)].copy()
toronto_geo["ward_name"] = toronto_geo["Geo Name"].map(TORONTO_WARDS)

assert len(toronto_geo) == 25, f"expected 25 Toronto wards, found {len(toronto_geo)}"
print(f"Found all {len(toronto_geo)} Toronto wards in the geography index")

toronto_geo[["Geo Name", "ward_name", "Line Number", "End Line"]].sort_values("Line Number")

Found all 25 Toronto wards in the geography index


,Geo Name,ward_name,Line Number,End Line
123,Beaches--East York,Beaches-East York,323615,326246.0
134,Davenport,Davenport,352556,355187.0
135,Don Valley East,Don Valley East,355187,357818.0
136,Don Valley North,Don Valley North,357818,360449.0
137,Don Valley West,Don Valley West,360449,363080.0
140,Eglinton--Lawrence,Eglinton-Lawrence,368342,370973.0
143,Etobicoke Centre,Etobicoke Centre,376235,378866.0
144,Etobicoke--Lakeshore,Etobicoke-Lakeshore,378866,381497.0
145,Etobicoke North,Etobicoke North,381497,384128.0
197,Parkdale--High Park,Parkdale-High Park,518309,520940.0


## Extract Toronto ward rows, filtered to the characteristics we care about

The main data file is a single ~185MB CSV covering every federal riding in
Canada. Rather than seeking to each riding's row range individually, we read
it once with only the columns we need and filter directly on `GEO_NAME`
(the riding name) and `CHARACTERISTIC_ID` (the variables from `ID_RANGES`
above) — the same one-pass approach used in the ADA extract.

In [4]:
# CELL 4 — Read the main file once, filter to Toronto wards and our characteristic IDs

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Flatten ID_RANGES into a set of valid CHARACTERISTIC_IDs
valid_ids = set()
for start, end in ID_RANGES:
    valid_ids.update(range(start, end + 1))

required_cols = [
    "GEO_NAME",
    "CHARACTERISTIC_ID",
    "CHARACTERISTIC_NAME",
    "C1_COUNT_TOTAL",
    "C10_RATE_TOTAL",
]

print("Reading census data file...")
df = pd.read_csv(
    MAIN_FILE,
    low_memory=False,
    encoding="latin",
    usecols=required_cols,
    dtype={"GEO_NAME": str},
)

df["CHARACTERISTIC_ID"] = pd.to_numeric(df["CHARACTERISTIC_ID"], errors="coerce")

df_filtered = df[
    (df["CHARACTERISTIC_ID"].isin(valid_ids)) &
    (df["GEO_NAME"].isin(TORONTO_WARDS))
].copy()

print(f"Extracted {len(df_filtered)} rows across {df_filtered['GEO_NAME'].nunique()} Toronto wards")

# Attach the ward_name / ward_code join keys and a human-readable characteristic code
df_filtered["ward_name"] = df_filtered["GEO_NAME"].map(TORONTO_WARDS)
df_filtered["ward_code"] = df_filtered["ward_name"].map(WARD_CODES)
df_filtered["CHARACTERISTIC_CODE"] = df_filtered["CHARACTERISTIC_ID"].map(CHARACTERISTIC_CODES)

output_cols = [
    "ward_code", "ward_name", "GEO_NAME",
    "CHARACTERISTIC_ID", "CHARACTERISTIC_NAME", "CHARACTERISTIC_CODE",
    "C1_COUNT_TOTAL", "C10_RATE_TOTAL",
]
df_filtered = df_filtered[output_cols]

df_filtered.head()

Reading census data file...


Extracted 4600 rows across 25 Toronto wards


,ward_code,ward_name,GEO_NAME,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,CHARACTERISTIC_CODE,C1_COUNT_TOTAL,C10_RATE_TOTAL
323613,19,Beaches-East York,Beaches--East York,1,"Population, 2021",pop_2021,109359.0,NaN
323614,19,Beaches-East York,Beaches--East York,2,"Population, 2016",pop_2016,109468.0,NaN
323615,19,Beaches-East York,Beaches--East York,3,"Population percentage change, 2016 to 2021",pop_pct_change,-0.1,-0.1
323618,19,Beaches-East York,Beaches--East York,6,Population density per square kilometre,pop_density,6531.2,6531.2
323646,19,Beaches-East York,Beaches--East York,34,Total - Distribution (%) of the population by ...,age_total_dist,100.0,100.0


## Save per-ward CSVs, then merge into one final ward-level CSV

In [5]:
# CELL 5 — Save each ward to its own file, named by ward_code

print("Saving individual ward files...")
for ward_code in tqdm(sorted(df_filtered["ward_code"].unique())):
    output_file = os.path.join(OUTPUT_DIR, f"{ward_code}.csv")
    df_ward = df_filtered[df_filtered["ward_code"] == ward_code][output_cols]
    df_ward.to_csv(output_file, index=False)

print("Per-ward extraction complete!")

Saving individual ward files...


  0%|          | 0/25 [00:00<?, ?it/s]

100%|██████████| 25/25 [00:00<00:00, 431.17it/s]

Per-ward extraction complete!


In [6]:
# CELL 6 — Merge all Toronto ward CSVs into one final CSV

csv_files = sorted(os.listdir(OUTPUT_DIR))
write_header = True

with open(FINAL_OUTPUT, "w", encoding="latin") as outfile:
    for fname in tqdm(csv_files):
        fpath = os.path.join(OUTPUT_DIR, fname)
        with open(fpath, "r", encoding="latin") as infile:
            if write_header:
                outfile.write(infile.read())
                write_header = False
            else:
                next(infile)  # skip header
                outfile.write(infile.read())

print(f"Final merged CSV saved: {FINAL_OUTPUT}")

  0%|          | 0/25 [00:00<?, ?it/s]

100%|██████████| 25/25 [00:00<00:00, 677.58it/s]

Final merged CSV saved: ../../data/census/wards/toronto-wards.csv


## Sanity checks

Confirm the merged output has exactly 25 wards, every characteristic ID we
expect, and that a couple of well-known figures (e.g. total 2021 population)
land in a plausible range.

In [7]:
# CELL 7 — Verify the merged output

final_df = pd.read_csv(FINAL_OUTPUT, encoding="latin")

assert final_df["ward_code"].nunique() == 25, final_df["ward_code"].nunique()
assert set(final_df["ward_name"].unique()) == set(TORONTO_WARDS.values())
assert set(final_df["CHARACTERISTIC_ID"].unique()) == valid_ids

print(f"Rows: {len(final_df)}")
print(f"Wards: {final_df['ward_code'].nunique()}")
print(f"Characteristics: {final_df['CHARACTERISTIC_ID'].nunique()}")

pop_2021 = final_df[final_df["CHARACTERISTIC_CODE"] == "pop_2021"][["ward_code", "ward_name", "C1_COUNT_TOTAL"]]
pop_2021 = pop_2021.sort_values("C1_COUNT_TOTAL", ascending=False)

print(f"\nTotal Toronto population (sum of 25 wards): {pop_2021['C1_COUNT_TOTAL'].sum():,.0f}")
pop_2021

Rows: 4600
Wards: 25
Characteristics: 184

Total Toronto population (sum of 25 wards): 2,794,356


,ward_code,ward_name,C1_COUNT_TOTAL
368,3,Etobicoke-Lakeshore,141751.0
1656,10,Spadina-Fort York,136213.0
2208,13,Toronto Centre,119901.0
184,2,Etobicoke Centre,118483.0
3128,18,Willowdale,118218.0
2024,12,Toronto-St. Paul's,116953.0
736,5,York South-Weston,116757.0
0,1,Etobicoke North,116003.0
1288,8,Eglinton-Lawrence,115832.0
2944,17,Don Valley North,113663.0


## Bonus: NOC/NAICS labour-force data, interpolated to wards

`data/census/noc-naics` holds a separate StatCan extract -- creative-occupation (NOC),
cultural-industry (NAICS), and arts-major (CIP) labour-force slices -- that isn't part
of the main `98-401-X2021010` product above, and is already prepared at the
**census-tract** level with tract polygons attached
(`data/census/ada-wide/toronto-tract-noc-naics.geojson`). `analysis/census/interpolate_noc_naics.ipynb`
already interpolates this from tracts to ADAs, but it does so by area-weighting the
*percentage* fields directly (`tobler`'s `intensive_variables`), which implicitly assumes
population is spread evenly across each tract's area.

That assumption gets worse the bigger the target unit is, and wards are much bigger than
ADAs -- a ward can span tracts that vary a lot in population density (a dense downtown
tract vs. a sprawling low-density one). Splitting a *percentage* by raw area share would let
a large, sparse tract corner of a ward outweigh a small, dense tract that actually holds
most of the people, or vice versa.

Instead, for wards we interpolate the underlying **counts** (extensive: population and each
labour count, which scale with population) by area overlap -- same assumption of uniform
density *within* a tract, but only used to move population and labour counts, not rates --
sum the allocated counts per ward, and only then divide to get each ward's percentage. That
makes the resulting rate population-weighted: a ward's `labour_creatives_pct` reflects where
the actual people (and creatives) are estimated to live, not which tract's polygon happens
to cover more of the ward's land.

In [8]:
# CELL 8 — Load tract-level NOC/NAICS data and ward polygons

noc_naics_gdf = gpd.read_file(NOC_NAICS_TRACT_GEOJSON)
print(f"NOC/NAICS census tracts loaded: {len(noc_naics_gdf)}")

# a couple of tracts have no data (suppressed / outside the covered area) -- drop them
# rather than interpolating a NaN into every ward that overlaps them
n_missing = noc_naics_gdf[NOC_NAICS_COUNT_FIELDS].isna().any(axis=1).sum()
noc_naics_gdf = noc_naics_gdf.dropna(subset=NOC_NAICS_COUNT_FIELDS, how="any").reset_index(drop=True)
print(f"Dropped {n_missing} tract(s) with missing counts, {len(noc_naics_gdf)} remain")

wards_gdf = gpd.read_file(WARDS_GEOJSON)

# the ward file also carries the boundary line geometries for the same 25 wards
# (a rendering artifact) -- keep only the polygon rows
wards_gdf = wards_gdf[wards_gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].reset_index(drop=True)
assert len(wards_gdf) == 25, f"expected 25 ward polygons, found {len(wards_gdf)}"

wards_gdf["geometry"] = wards_gdf.geometry.make_valid()
noc_naics_gdf["geometry"] = noc_naics_gdf.geometry.make_valid()

wards_gdf.head()

NOC/NAICS census tracts loaded: 582
Dropped 2 tract(s) with missing counts, 580 remain


,ward_code,ward_name,geometry
0,07,Humber River-Black Creek,"MULTIPOLYGON (((-79.49105 43.7635, -79.49008 4..."
1,06,York Centre,"MULTIPOLYGON (((-79.44043 43.7634, -79.43998 4..."
2,18,Willowdale,"MULTIPOLYGON (((-79.39449 43.76157, -79.39461 ..."
3,11,University-Rosedale,"MULTIPOLYGON (((-79.39004 43.6905, -79.39004 4..."
4,19,Beaches-East York,"MULTIPOLYGON (((-79.29864 43.71515, -79.29837 ..."


In [9]:
# CELL 9 — Population-weighted areal interpolation: tract -> ward
#
# 1. Reproject to a local projected CRS for accurate area math
# 2. Overlay tracts with wards to get the intersection pieces
# 3. weight = piece_area / original_tract_area
# 4. allocated_count = weight * tract_count, for every count field (population + each labour count)
# 5. Sum allocated counts by ward -- this is the population-weighted step, since the
#    thing being split by area is a count, not a rate
# 6. Recompute each pct field as allocated_count / allocated_population

proj_crs = wards_gdf.estimate_utm_crs()
tracts_proj = noc_naics_gdf.to_crs(proj_crs)
wards_proj = wards_gdf.to_crs(proj_crs)

tracts_slim = tracts_proj[["census_tract", *NOC_NAICS_COUNT_FIELDS, "geometry"]].copy()
tracts_slim["source_area"] = tracts_slim.geometry.area

pieces = gpd.overlay(
    tracts_slim,
    wards_proj[[WARD_NAME_COL, "geometry"]],
    how="intersection",
    keep_geom_type=True,
)

pieces["piece_area"] = pieces.geometry.area
pieces["area_weight"] = pieces["piece_area"] / pieces["source_area"]

for field in NOC_NAICS_COUNT_FIELDS:
    pieces[field] = pieces["area_weight"] * pieces[field]

ward_noc_naics = pieces.groupby(WARD_NAME_COL, as_index=False)[NOC_NAICS_COUNT_FIELDS].sum()

# recompute percentages at the ward level (population-weighted, not area-weighted)
for count_field, pct_field in NOC_NAICS_PCT_FIELDS.items():
    ward_noc_naics[pct_field] = (
        100 * ward_noc_naics[count_field] / ward_noc_naics["pop_2021_count"]
    ).round(1)

ward_noc_naics["ward_code"] = ward_noc_naics[WARD_NAME_COL].map(WARD_CODES)
ward_noc_naics = ward_noc_naics.sort_values("ward_code").reset_index(drop=True)

ward_noc_naics.head()

,ward_name,pop_2021_count,labour_creatives_count,labour_cultural_workers_count,labour_cultural_industries_count,labour_independent_artists_count,labour_arts_major_count,labour_creatives_pct,labour_cultural_workers_pct,labour_cultural_industries_pct,labour_independent_artists_pct,labour_arts_major_pct,ward_code
0,Etobicoke North,97534.650127,344.579891,700.252677,825.730870,124.498434,1761.983531,0.4,0.7,0.8,0.1,1.8,01
1,Etobicoke Centre,99390.021043,975.371948,1832.418101,1533.500842,337.752790,3960.920034,1.0,1.8,1.5,0.3,4.0,02
2,Etobicoke-Lakeshore,106707.987019,2106.414632,3616.235401,2841.024347,621.277921,5887.014148,2.0,3.4,2.7,0.6,5.5,03
3,Parkdale-High Park,88396.783523,4011.710292,6657.340576,5592.956218,1875.486765,9514.236102,4.5,7.5,6.3,2.1,10.8,04
4,York South-Weston,97190.081660,887.190626,1538.225646,1390.134961,258.050694,2593.500159,0.9,1.6,1.4,0.3,2.7,05


In [10]:
# CELL 10 — Sanity checks + save

assert len(ward_noc_naics) == 25, f"expected 25 wards, got {len(ward_noc_naics)}"
assert ward_noc_naics["ward_code"].notna().all(), "unmapped ward_name in NOC/NAICS interpolation"

# reconstruction check: interpolated population should sum back to ~ the tract total
# (small diffs are OK -- a handful of tracts straddle city limits and lose a sliver
# outside the 25 wards; large diffs would flag a join/overlay bug)
tract_total_pop = noc_naics_gdf["pop_2021_count"].sum()
ward_total_pop = ward_noc_naics["pop_2021_count"].sum()
pct_diff = abs(ward_total_pop - tract_total_pop) / tract_total_pop
print(f"Tract population total: {tract_total_pop:,.0f}")
print(f"Ward (interpolated) population total: {ward_total_pop:,.0f}  (diff: {pct_diff:.2%})")
assert pct_diff < 0.02, "interpolated ward population diverges too much from tract total"

for pct_field in NOC_NAICS_PCT_FIELDS.values():
    assert ward_noc_naics[pct_field].between(0, 100).all(), f"{pct_field} out of [0, 100] range"

output_cols = ["ward_code", WARD_NAME_COL, *NOC_NAICS_COUNT_FIELDS, *NOC_NAICS_PCT_FIELDS.values()]
ward_noc_naics[output_cols].to_csv(NOC_NAICS_OUTPUT, index=False)
print(f"\nSaved: {NOC_NAICS_OUTPUT}")

ward_noc_naics[output_cols].sort_values("labour_creatives_pct", ascending=False)

Tract population total: 2,363,075
Ward (interpolated) population total: 2,362,498  (diff: 0.02%)

Saved: ../../data/census/wards/toronto-wards-noc-naics.csv


,ward_code,ward_name,pop_2021_count,labour_creatives_count,labour_cultural_workers_count,labour_cultural_industries_count,labour_independent_artists_count,labour_arts_major_count,labour_creatives_pct,labour_cultural_workers_pct,labour_cultural_industries_pct,labour_independent_artists_pct,labour_arts_major_pct
8,09,Davenport,92138.973002,4276.465492,6916.875315,5213.862006,1657.792164,8940.906133,4.6,7.5,5.7,1.8,9.7
3,04,Parkdale-High Park,88396.783523,4011.710292,6657.340576,5592.956218,1875.486765,9514.236102,4.5,7.5,6.3,2.1,10.8
10,11,University-Rosedale,91192.896497,3915.530363,6277.367539,4930.474280,1946.288483,9583.583486,4.3,6.9,5.4,2.1,10.5
9,10,Spadina-Fort York,130271.658243,5176.593032,8080.120860,5964.617585,1915.051699,9634.419489,4.0,6.2,4.6,1.5,7.4
13,14,Toronto-Danforth,89241.084289,3593.129929,6015.286065,4995.531279,1675.442475,8118.323971,4.0,6.7,5.6,1.9,9.1
12,13,Toronto Centre,107399.762122,3449.834423,5629.014408,4149.074472,1324.892662,8292.011680,3.2,5.2,3.9,1.2,7.7
11,12,Toronto-St. Paul's,100326.079223,3118.703680,5134.568132,3887.912833,1436.582380,8428.008044,3.1,5.1,3.9,1.4,8.4
18,19,Beaches-East York,88150.198659,2532.282283,4406.228565,3920.450063,1137.271336,6491.065025,2.9,5.0,4.4,1.3,7.4
2,03,Etobicoke-Lakeshore,106707.987019,2106.414632,3616.235401,2841.024347,621.277921,5887.014148,2.0,3.4,2.7,0.6,5.5
14,15,Don Valley West,82456.835794,1341.582053,2397.864048,1563.312610,413.021556,4530.936246,1.6,2.9,1.9,0.5,5.5
